In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path().resolve().parents[0]

METADATA_PATH = PROJECT_ROOT / "data" / "interim" / "trap_metadata.csv"
TRAP_COUNT_PATH = PROJECT_ROOT / "data" / "processed" / "stopwhitefly_trap_weekly_counts_with_metadata.csv"

metadata_df = pd.read_csv(METADATA_PATH)
trap_count_df = pd.read_csv(TRAP_COUNT_PATH)

print("Metadata shape:", metadata_df.shape)
print("Trap-count shape:", trap_count_df.shape)

print("\nMetadata columns:")
print(metadata_df.columns.tolist())

print("\nTrap-count columns:")
print(trap_count_df.columns.tolist())

print("\nMetadata dtypes:")
print(metadata_df.dtypes)

print("\nTrap-week dtypes:")
print(trap_count_df.dtypes)

print("\nMetadata missing values:")
print(metadata_df.isna().sum())

print("\nTrap-count missing values:")
print(trap_count_df.isna().sum())

print("\nUnique metadata sites:", metadata_df["site_id"].nunique())
print("Unique trap-week sites:", trap_count_df["site_id"].nunique())

print("\nTrap years:")
print(sorted(trap_count_df["year"].unique()))

print("\nLatitude range:", metadata_df["latitude"].min(), metadata_df["latitude"].max())
print("Longitude range:", metadata_df["longitude"].min(), metadata_df["longitude"].max())

metadata_df.head()

Metadata shape: (31, 7)
Trap-count shape: (6121, 11)

Metadata columns:
['site_id', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']

Trap-count columns:
['site_id', 'year', 'plotted_date', 'date_collected', 'whitefly_count', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report']

Metadata dtypes:
site_id           int64
trap_label          str
latitude        float64
longitude       float64
status              str
first_report        str
last_report         str
dtype: object

Trap-week dtypes:
site_id             int64
year                int64
plotted_date          str
date_collected        str
whitefly_count      int64
trap_label            str
latitude          float64
longitude         float64
status                str
first_report          str
last_report           str
dtype: object

Metadata missing values:
site_id         0
trap_label      0
latitude        0
longitude       0
status          0
first_report    0
last_repor

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [3]:
import geopandas as gpd

In [ ]:
trap_gdf = gpd.GeoDataFrame(
    metadata_df.copy(),
    geometry=gpd.points_from_xy(metadata_df["longitude"], metadata_df["latitude"]),
    crs="EPSG:4326"  # geographic CRS (points are in geographic degrees)
)

print("Original CRS:", trap_gdf.crs)
print("Shape:", trap_gdf.shape)

print("\nGeometry type counts:")
print(trap_gdf.geometry.geom_type.value_counts())

print("\nBounds in latitude/longitude CRS:")
print(trap_gdf.total_bounds)

trap_gdf.head()

Original CRS: EPSG:4326
Shape: (31, 8)

Geometry type counts:
Point    31
Name: count, dtype: int64

Bounds in latitude/longitude CRS:
[-84.895381  30.798771 -83.405028  31.557827]


,site_id,trap_label,latitude,longitude,status,first_report,last_report,geometry
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22,POINT (-83.46501 31.46603)
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22,POINT (-83.40503 31.46509)
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22,POINT (-83.48018 31.55783)
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15,POINT (-83.57304 31.54121)
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15,POINT (-83.66288 31.52428)


#### Reproject into a meter-based CRS

In [7]:
# Reproject trap points from geographic CRS to projected CRS (UTM Zone 17N)
PROJECTED_CRS = "EPSG:26917"

trap_gdf_utm = trap_gdf.to_crs(PROJECTED_CRS)

print("Original CRS:", trap_gdf.crs)
print("Projected CRS:", trap_gdf_utm.crs)

print("\nBounds in projected CRS/meters:")
print(trap_gdf_utm.total_bounds)

print("\nFirst few projected geometries:")
print(trap_gdf_utm[["site_id", "trap_label", "geometry"]].head())

Original CRS: EPSG:4326
Projected CRS: EPSG:26917

Bounds in projected CRS/meters:
[ 129436.91781993 3410393.01188487  271492.41530177 3494093.91594691]

First few projected geometries:
   site_id trap_label                        geometry
0    29221   Trap 101  POINT (265794.327 3483882.818)
1    29222   Trap 102  POINT (271492.415 3483651.729)
2    29224   Trap 104  POINT (264582.565 3494093.916)
3    29225   Trap 105  POINT (255722.656 3492455.271)
4    29226   Trap 106  POINT (247145.224 3490781.095)


#### Create 500 m and 1 km buffers

In [10]:
# create buffer layers in projected CRS
trap_buffers_500m = trap_gdf_utm.copy()
trap_buffers_500m["buffer_radius_m"] = 500
trap_buffers_500m["geometry"] = trap_buffers_500m.geometry.buffer(500)

trap_buffers_1km = trap_gdf_utm.copy()
trap_buffers_1km["buffer_radius_m"] = 1000
trap_buffers_1km["geometry"] = trap_buffers_1km.geometry.buffer(1000)

print("500 m buffer CRS:", trap_buffers_500m.crs)
print("1 km buffer CRS:", trap_buffers_1km.crs)

print("\n500 m buffer shape:", trap_buffers_500m.shape)
print("1 km buffer shape:", trap_buffers_1km.shape)

print("\nGeometry types:")
print("500 m:")
print(trap_buffers_500m.geometry.geom_type.value_counts())
print("\n1 km:")
print(trap_buffers_1km.geometry.geom_type.value_counts())

print("\nApproximate buffer areas in square meters:")
print("500 m buffer area, first site:", trap_buffers_500m.geometry.area.iloc[0])
print("1 km buffer area, first site:", trap_buffers_1km.geometry.area.iloc[0])

500 m buffer CRS: EPSG:26917
1 km buffer CRS: EPSG:26917

500 m buffer shape: (31, 9)
1 km buffer shape: (31, 9)

Geometry types:
500 m:
Polygon    31
Name: count, dtype: int64

1 km:
Polygon    31
Name: count, dtype: int64

Approximate buffer areas in square meters:
500 m buffer area, first site: 784137.1226364922
1 km buffer area, first site: 3136548.490546164


#### Combining the two buffer layers

In [11]:
trap_buffers_all = pd.concat(
    [trap_buffers_500m, trap_buffers_1km],
    ignore_index=True
)

print("Combined buffer shape:", trap_buffers_all.shape)
print("\nBuffer radius counts:")
print(trap_buffers_all["buffer_radius_m"].value_counts().sort_index())

print("\nCRS:", trap_buffers_all.crs)

print("\nGeometry type counts:")
print(trap_buffers_all.geometry.geom_type.value_counts())

trap_buffers_all[["site_id", "trap_label", "buffer_radius_m", "geometry"]].head()

Combined buffer shape: (62, 9)

Buffer radius counts:
buffer_radius_m
500     31
1000    31
Name: count, dtype: int64

CRS: EPSG:26917

Geometry type counts:
Polygon    62
Name: count, dtype: int64


,site_id,trap_label,buffer_radius_m,geometry
0,29221,Trap 101,500,"POLYGON ((266294.327 3483882.818, 266291.919 3..."
1,29222,Trap 102,500,"POLYGON ((271992.415 3483651.729, 271990.008 3..."
2,29224,Trap 104,500,"POLYGON ((265082.565 3494093.916, 265080.158 3..."
3,29225,Trap 105,500,"POLYGON ((256222.656 3492455.271, 256220.248 3..."
4,29226,Trap 106,500,"POLYGON ((247645.224 3490781.095, 247642.816 3..."


#### Save buffer layer for reuse

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "geospatial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUFFER_OUTPUT_PATH = OUTPUT_DIR / "trap_buffers_500m_1km.gpkg"

# Save as a GeoPackage (.gpkg)
trap_buffers_all.to_file(BUFFER_OUTPUT_PATH, layer="trap_buffers", driver="GPKG")

print("Saved buffer layer to:", BUFFER_OUTPUT_PATH)

Saved buffer layer to: /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/interim/geospatial/trap_buffers_500m_1km.gpkg


#### Validation to ensure saved GeoPackage can be read correctly

In [13]:
buffers_check = gpd.read_file(BUFFER_OUTPUT_PATH, layer="trap_buffers")

print("Shape of saved file:", buffers_check.shape)
print("CRS of saved file:", buffers_check.crs)

print("\nBuffer radius counts:")
print(buffers_check["buffer_radius_m"].value_counts().sort_index())

print("\nGeometry type counts:")
print(buffers_check.geometry.geom_type.value_counts())

print("\nColumns:")
print(buffers_check.columns.to_list())

buffers_check[["site_id", "trap_label", "buffer_radius_m", "geometry"]].head()

Shape of saved file: (62, 9)
CRS of saved file: EPSG:26917

Buffer radius counts:
buffer_radius_m
500     31
1000    31
Name: count, dtype: int64

Geometry type counts:
Polygon    62
Name: count, dtype: int64

Columns:
['site_id', 'trap_label', 'latitude', 'longitude', 'status', 'first_report', 'last_report', 'buffer_radius_m', 'geometry']


,site_id,trap_label,buffer_radius_m,geometry
0,29221,Trap 101,500,"POLYGON ((266294.327 3483882.818, 266291.919 3..."
1,29222,Trap 102,500,"POLYGON ((271992.415 3483651.729, 271990.008 3..."
2,29224,Trap 104,500,"POLYGON ((265082.565 3494093.916, 265080.158 3..."
3,29225,Trap 105,500,"POLYGON ((256222.656 3492455.271, 256220.248 3..."
4,29226,Trap 106,500,"POLYGON ((247645.224 3490781.095, 247642.816 3..."


#### Next, we download the land cover raster for the year 2025 for southern Georgia from MRLC website and save the files in the ~/data/raw/nlcd/2025 folder. 

#### Inspect the raster

In [14]:
from pathlib import Path
import rasterio

In [15]:
NLCD_2025_PATH = PROJECT_ROOT / "data" / "raw" / "nlcd" / "2025" / "nlcd_landcover_2025.tif"

with rasterio.open(NLCD_2025_PATH) as src:
    print("Raster path:", NLCD_2025_PATH)
    print("CRS:", src.crs)
    print("Width, height:", src.width, src.height)
    print("Number of bands:", src.count)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Nodata:", src.nodata)
    print("Data type:", src.dtypes)

Raster path: /Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/data/raw/nlcd/2025/nlcd_landcover_2025.tif
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Width, height: 14847 11731
Number of bands: 1
Resolution: (30.0, 30.0)
Bounds: BoundingBox(left=958635.0, bottom=830025.0, right=1404045.0, top=1181955.0)
Nodata: 250.0
Data type: ('uint8',)


#### Verify overlap between buffers and NLCD raster

In [16]:
import geopandas as gpd
import rasterio
from shapely.geometry import box

In [ ]:
BUFFER_PATH = PROJECT_ROOT / "data" / "interim" / "geospatial" / "trap_buffers_500m_1km.gpkg"

# Read buffer layer
buffers_gdf = gpd.read_file(BUFFER_PATH, layer="trap_buffers")

# Open the NLCD raster
with rasterio.open(NLCD_2025_PATH) as src:
    nlcd_crs = src.crs
    nlcd_bounds = src.bounds

# Reproject buffers to NLCD CRS
buffers_nlcd_crs = buffers_gdf.to_crs(nlcd_crs)

# Create raster footprint polygon
nlcd_bounds_gdf = gpd.GeoDataFrame(
    {"name": ["nlcd_bounds"]},
    geometry=[box(nlcd_bounds.left, nlcd_bounds.bottom, nlcd_bounds.right, nlcd_bounds.top)],
    crs=nlcd_crs
)

print("Original buffer CRS:", buffers_gdf.crs)
print("NLCD CRS:", nlcd_crs)
print("Reprojected buffer CRS:", buffers_nlcd_crs.crs)

print("\nBuffer bounds in NLCD CRS:")
print(buffers_nlcd_crs.total_bounds)

print("\nNLCD raster bounds:")
print(nlcd_bounds)

print("\nDo all buffers intersect NLCD raster bounds?")
print(buffers_nlcd_crs.intersects(nlcd_bounds_gdf.geometry.iloc[0]).all())

print("\nNumber of buffers intersecting NLCD raster bounds:")
print(buffers_nlcd_crs.intersects(nlcd_bounds_gdf.geometry.iloc[0]).sum(), "of", len(buffers_nlcd_crs))

Original buffer CRS: EPSG:26917
NLCD CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Reprojected buffer CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["lon

#### First extraction function: one raster year, all buffers

In [18]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

In [ ]:
def extract_nlcd_class_counts_for_buffers(
        raster_path,
        buffers_gdf,
        site_id_col="site_id",
        trap_label_col="trap_label",
        buffer_radius_col="buffer_radius_m"
):
    """
    Extract NLCD class pixel counts for each buffer polygon.

    Parameters:
    ----------
    raster_path: path_like
        Path to NLCD land-cover raster.
    buffers_gdf: geopandas.GeoDataFrame
        Buffer polygons. Can be in any CRS; function reprojects to raster CRS.
    site_id_col: str
        Site ID column name
    trap_label_col: str
        Trap label column name
    buffer_radius_col: str
        Buffer radius column name

    Returns:
    -------
    pandas.DataFrame
        Long table with one row per site-buffer-class
    """
    records = []

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

        buffers_for_raster = buffers_gdf.to_crs(raster_crs)

        for _, row in buffers_for_raster.iterrows():
            geom = [row.geometry] # mask() requires a list of geometries

            # Clip the raster to the polygon
            out_image, _ = mask(
                src,                # the raster
                geom,               # the polygon to clip with
                crop=True,          # output only the area inside the polygon
                filled=True,        # fill areas outside the polygon with nodata
                nodata=nodata       # use the raster's nodata value
            )

            # flatten the pixel values in the first and only band in the out_image
            values = out_image[0].ravel()  

            # Remove nodata values
            if nodata is not None:
                values = values[values != nodata]

            # Remove zeros if present as background/unclassified
            values = values[values != 0]

            unique_values, counts = np.unique(values, return_counts=True)

            for class_value, count in zip(unique_values, counts):
                records.append(
                    {
                        site_id_col: row[site_id_col],
                        trap_label_col: row[trap_label_col],
                        buffer_radius_col: row[buffer_radius_col],
                        "nlcd_class": int(class_value),
                        "pixel_count": int(count)
                    }
                )

    return pd.DataFrame(records)

In [22]:
nlcd_counts_2025 = extract_nlcd_class_counts_for_buffers(
    raster_path=NLCD_2025_PATH,
    buffers_gdf=buffers_gdf
)

print("NLCD class count table shape:", nlcd_counts_2025.shape)
print("\nFirst rows:")
display(nlcd_counts_2025.head(20))

print("\nUnique NLCD classes found:")
print(sorted(nlcd_counts_2025["nlcd_class"].unique()))

print("\nNumber of site-buffer combinations:")
print(nlcd_counts_2025[["site_id", "buffer_radius_m"]].drop_duplicates().shape[0])

NLCD class count table shape: (639, 5)

First rows:


,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count
0,29221,Trap 101,500,11,10
1,29221,Trap 101,500,21,107
2,29221,Trap 101,500,22,70
3,29221,Trap 101,500,42,149
4,29221,Trap 101,500,52,15
5,29221,Trap 101,500,71,8
6,29221,Trap 101,500,82,402
7,29221,Trap 101,500,90,108
8,29221,Trap 101,500,95,2
9,29222,Trap 102,500,11,26



Unique NLCD classes found:
[np.int64(11), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(31), np.int64(41), np.int64(42), np.int64(43), np.int64(52), np.int64(71), np.int64(81), np.int64(82), np.int64(90), np.int64(95)]

Number of site-buffer combinations:
62


#### Convert the long class-count table into grouped biological features

In [23]:
NLCD_CLASS_GROUPS = {
    "water": [11],
    "developed": [21, 22, 23, 24],
    "barren": [31],
    "forest": [41, 42, 43],
    "shrub_scrub": [52],
    "grassland_herbaceous": [71],
    "pasture_hay": [81],
    "cultivated_crops": [82],
    "wetlands": [90, 95],
}

def add_nlcd_group_labels(counts_df):
    """
    Add broad NLCD group labels to a long class-count table
    """
    class_to_group = {}

    for group_name, class_codes in NLCD_CLASS_GROUPS.items():
        for code in class_codes:
            class_to_group[code] = group_name

    labeled_df = counts_df.copy()
    labeled_df["landcover_group"] = labeled_df["nlcd_class"].map(class_to_group)
    labeled_df["landcover_group"] = labeled_df["landcover_group"].fillna("other")

    return labeled_df

In [24]:
nlcd_counts_labeled_2025 = add_nlcd_group_labels(nlcd_counts_2025)

print("Labeled counts shape:", nlcd_counts_labeled_2025.shape)

print("\nGroup counts:")
print(nlcd_counts_labeled_2025["landcover_group"].value_counts())

display(nlcd_counts_labeled_2025.head(20))

Labeled counts shape: (639, 6)

Group counts:
landcover_group
developed               160
wetlands                109
forest                  102
cultivated_crops         62
shrub_scrub              55
water                    53
grassland_herbaceous     50
pasture_hay              43
barren                    5
Name: count, dtype: int64


,site_id,trap_label,buffer_radius_m,nlcd_class,pixel_count,landcover_group
0,29221,Trap 101,500,11,10,water
1,29221,Trap 101,500,21,107,developed
2,29221,Trap 101,500,22,70,developed
3,29221,Trap 101,500,42,149,forest
4,29221,Trap 101,500,52,15,shrub_scrub
5,29221,Trap 101,500,71,8,grassland_herbaceous
6,29221,Trap 101,500,82,402,cultivated_crops
7,29221,Trap 101,500,90,108,wetlands
8,29221,Trap 101,500,95,2,wetlands
9,29222,Trap 102,500,11,26,water


In [27]:
def summarize_nlcd_group_fractions(labeled_counts_df):
    """
    Convert NLCD class counts into broad land-cover group fractions for each 
    buffer site combination.
    """
    group_counts = (
        labeled_counts_df
        .groupby(["site_id", "trap_label", "buffer_radius_m", "landcover_group"], as_index=False)
        ["pixel_count"]
        .sum()
    )

    total_counts = (
        group_counts
        .groupby(["site_id", "trap_label", "buffer_radius_m"], as_index=False)
        ["pixel_count"]
        .sum()
        .rename(columns={"pixel_count": "total_valid_pixels"})
    )

    group_fractions = group_counts.merge(
        total_counts,
        on=["site_id", "trap_label", "buffer_radius_m"],
        how="left"
    )

    group_fractions["fraction"] = (
        group_fractions["pixel_count"] / group_fractions["total_valid_pixels"]
    )

    return group_fractions

In [28]:
nlcd_group_fractions_2025 = summarize_nlcd_group_fractions(nlcd_counts_labeled_2025)

print("Group fraction table shape:", nlcd_group_fractions_2025.shape)

print("\nFirst rows:")
display(nlcd_group_fractions_2025.head(20))

print("\nFraction sum check:")
fraction_check = (
    nlcd_group_fractions_2025
    .groupby(["site_id", "buffer_radius_m"])["fraction"]
    .sum()
    .reset_index(name="fraction_sum")
)

display(fraction_check.head())
print(fraction_check["fraction_sum"].describe())

Group fraction table shape: (454, 7)

First rows:


,site_id,trap_label,buffer_radius_m,landcover_group,pixel_count,total_valid_pixels,fraction
0,29221,Trap 101,500,cultivated_crops,402,871,0.461538
1,29221,Trap 101,500,developed,177,871,0.203215
2,29221,Trap 101,500,forest,149,871,0.171068
3,29221,Trap 101,500,grassland_herbaceous,8,871,0.009185
4,29221,Trap 101,500,shrub_scrub,15,871,0.017222
5,29221,Trap 101,500,water,10,871,0.011481
6,29221,Trap 101,500,wetlands,110,871,0.126292
7,29221,Trap 101,1000,cultivated_crops,1742,3487,0.499570
8,29221,Trap 101,1000,developed,690,3487,0.197878
9,29221,Trap 101,1000,forest,448,3487,0.128477



Fraction sum check:


,site_id,buffer_radius_m,fraction_sum
0,29221,500,1.0
1,29221,1000,1.0
2,29222,500,1.0
3,29222,1000,1.0
4,29224,500,1.0


count    6.200000e+01
mean     1.000000e+00
std      1.421495e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: fraction_sum, dtype: float64
